# AgentCore Runtime Resource Management Operations

This notebook demonstrates comprehensive **AgentCore Runtime** lifecycle management using the AgentCoreRuntimeClient.

## What You'll Learn

- **Create**: Agent runtime resources with custom business logic and configurations
- **Read**: Get runtime details and status information
- **Update**: Modify runtime settings and configurations
- **Delete**: Clean up runtime resources
- **List**: Enumerate all runtime resources

## AgentCore Runtime Operations

You can find boto3 (Python) AgentCore control plane runtime operations on this page:

https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agentcore-control.html

* AgentCore Runtimes provide managed execution environments for AI agents, enabling secure deployment and invocation of agent workloads.

* The classes provided here under `src/agentic_platform/service/agentcore/runtime` wrap the AgentCore SDK service calls for runtime management.

* The goal of the AgentCore support in this project is to demonstrate how to wrap those AgentCore calls in resource management services and deploy those resource management services to AWS via Terraform.

* The runtime operations include:
    * create_agent_runtime: Create a new agent runtime with specified business logic and configuration
    * get_agent_runtime: Get information about a runtime by runtimeId
    * delete_agent_runtime: Delete the runtime resource
    * list_agent_runtimes: List all runtime resources
    * update_agent_runtime: Update runtime configuration

The example AgentCoreRuntimeClient in this project currently supports these operations:

- `CREATE` - Create new agent runtimes with custom business logic
- `GET` - Retrieve runtime details and status
- `UPDATE` - Update runtime configurations
- `DELETE` - Delete runtime resources
- `LIST` - List all runtimes
- `WAIT_FOR_READY` - Wait for runtime creation and readiness

## Key Concepts

**AgentCore Runtime**: A managed execution environment that hosts AI agent workloads. Runtimes handle agent lifecycle, scaling, and provide secure invocation endpoints.

**Business Logic**: Custom Python code that defines the agent's behavior and capabilities. The system uses Bedrock to automatically customize templates based on natural language descriptions.

**Protocol Type**: Supports HTTP and MCP (Model Context Protocol) for different communication patterns.

**Runtime Status**: Tracks the lifecycle state (CREATING, READY, FAILED, etc.) of the runtime deployment.

**Execution Role**: IAM role that provides the runtime with necessary AWS permissions.

## Prerequisites

- AWS credentials with AgentCore permissions
- AgentCore CLI installed and configured
- Docker available for containerization (if using custom containers)

### First let's look at the AgentCore Runtime Client from this project.

In [ ]:
%store -r tf_info
import os
os.environ['USER_POOL_ID'] = tf_info['cognito_user_pool_id']['value']
os.environ['USER_POOL_CLIENT_ID'] = tf_info['cognito_user_pool_client_id']['value']
os.environ['REGION'] = tf_info['aws_region']['value']


In [ ]:
# If I don't set __file__ then the load of agentcore_runtime_client.py into the notebook will fail.
__file__ = "../../src/agentic_platform/service/agentcore/runtime/client/agentcore_runtime_client.py"

In [ ]:
# %load ../../src/agentic_platform/service/agentcore/runtime/client/agentcore_runtime_client.py
"""
AgentCore Runtime Client for managing AWS Bedrock AgentCore Runtime resources.

This client provides methods to create, retrieve, update, delete, and list
AgentCore Runtime resources using the AWS Bedrock AgentCore Control Plane API.

Usage:
  - Create agent runtimes for hosting agent workloads
  - Manage runtime lifecycle and configuration
  - Handle runtime endpoints and versioning

Environment Variables:
  - REGION: AWS region for Bedrock AgentCore resources (default: us-west-2)
"""

import boto3
import logging
import os
import shutil
import subprocess
import time
from pathlib import Path
from typing import Optional
from uuid import uuid4

from agentic_platform.service.agentcore.types import (
    AgentRuntime,
    AgentRuntimeStatus,
    DeleteAgentRuntimeRequest,
    DeleteAgentRuntimeResponse,
    GetAgentRuntimeRequest,
    GetAgentRuntimeResponse,
    ListAgentRuntimesRequest,
    ListAgentRuntimesResponse,
    UpdateAgentRuntimeRequest,
    UpdateAgentRuntimeResponse,
)

# Configure logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Get environment variables
REGION = os.getenv('REGION', 'us-west-2')
USER_POOL_CLIENT_ID = os.getenv('USER_POOL_CLIENT_ID', None)
USER_POOL_ID = os.getenv('USER_POOL_ID', None)
if not (USER_POOL_CLIENT_ID and USER_POOL_ID):
    raise Exception('USER_POOL_ID and USER_POOL_CLIENT_ID environment variables must be set')

COGNITO_DISCOVERY_URL = f"https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration"

# Initialize AWS clients
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
agentcore_data_client = boto3.client('bedrock-agentcore', region_name=REGION)
bedrock_runtime_client = boto3.client('bedrock-runtime', region_name=REGION)

logger.info(f"os.getcwd() = {os.getcwd()}")
logger.info(f"os.path.abspath(__file__) = {os.path.abspath(__file__)}")
parent_dir = os.path.dirname(os.path.abspath(__file__))

# agentcore_deploy_template_path = f'{parent_dir}/.bedrock_agentcore.yaml.template'
# logger.info(f"agentcore_deploy_template_path = {agentcore_deploy_template_path}")


class AgentCoreRuntimeClient:
    @staticmethod
    def _run_subprocess_with_optional_streaming(args, stream_output=False):
        """
        Run subprocess with optional real-time streaming to stdout.
        
        Args:
            args: Command arguments list
            stream_output: Whether to stream output to stdout in real-time
            
        Returns:
            subprocess.CompletedProcess-like object with returncode, stdout, stderr
        """
        if stream_output:
            # Stream output in real-time while capturing for parsing
            import sys
            
            logger.info(f"Running command with streaming: {' '.join(args)}")
            
            process = subprocess.Popen(
                args,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,  # Merge stderr into stdout for unified streaming
                text=True,
                bufsize=1,  # Line buffered
                universal_newlines=True
            )
            
            stdout_lines = []
            
            # Stream output line by line
            for line in process.stdout:
                # Print to stdout for real-time visibility
                print(line.rstrip())
                sys.stdout.flush()
                
                # Also capture for later parsing
                stdout_lines.append(line)
            
            # Wait for process to complete
            return_code = process.wait()
            
            # Reconstruct stdout and stderr for compatibility
            stdout_content = ''.join(stdout_lines)
            
            # Create a result object similar to subprocess.run return
            class StreamedResult:
                def __init__(self, returncode, stdout, stderr=""):
                    self.returncode = returncode
                    self.stdout = stdout
                    self.stderr = stderr  # Empty since we merged stderr into stdout
            
            return StreamedResult(return_code, stdout_content, "")
            
        else:
            # Use normal subprocess.run for non-streaming mode
            return subprocess.run(args, capture_output=True, text=True)

    def create_agent_runtime(
        self,
        name: str,
        agent_description: str,
        entrypoint: Optional[str] = "entrypoint.py",
        ecr_repo_uri: Optional[str] = None,
        execution_role_arn: Optional[str] = None,
        protocol: Optional[str] = "HTTP",
        update_on_conflict: Optional[bool] = False
    ) -> str:
        """Create an Amazon Secure Agent Runtime using agentcore configure and agentcore launch.
        
        This method creates the agent runtime deployment by:
        1. Copying template files to a unique deployment directory
        2. Customizing business logic using Bedrock
        3. Running agentcore configure and launch with streaming output
        
        Args:
            name: The name of the agent runtime (required)
            agent_description: Natural language description (required)
            entrypoint: Entry point file name (default: entrypoint.py)
            ecr_repo_uri: ECR repository URI (optional) default None to create one automatically
            execution_role_arn: IAM role ARN (optional) default None to create one automatically
            protocol: Server protocol - HTTP or MCP (default: HTTP)
            update_on_conflict: Whether to update if runtime already exists (default: false)
        Returns:
            String with deployment details including ARN and runtime ID
        """
        try:
            # Sanitize the agent name to meet AgentCore requirements
            name = self.sanitize_name(name)
            logger.info(f"Creating agent runtime '{name}': {agent_description[:100]}...")
            
            # Step 1: Create deployment from existing template code
            project_root = Path(__file__).parent.parent.parent
            source_dir = project_root / "src" / "agentcore_mcp" / "agent_deployment_template"
            
            if not source_dir.exists():
                raise FileNotFoundError(f"Agent templates source directory not found: {source_dir}")
            
            # Step 2: Generate unique deployment directory under agent_deployments
            unique_suffix = uuid4().hex[:8]
            unique_dir = f"deployment_{unique_suffix}"
            deployment_dir = f"{project_root}/agent_deployments/{unique_dir}"

            try:
                # Copy template source to deployment directory
                shutil.copytree(source_dir, deployment_dir)
                logger.info(f"Copied agent files from {source_dir} to {deployment_dir}")
                
                # Step 2.1: Select appropriate template (SINGLE or MULTI)
                template_selection = self.select_entrypoint_template(agent_description)
                logger.info(f"Selected template: {template_selection}")
                
                # Step 2.2: Set up the correct entrypoint file
                if template_selection == 'MULTI':
                    shutil.move(f"{deployment_dir}/multi_entrypoint.py", f"{deployment_dir}/entrypoint.py")
                    logger.info("Using multi-agent template as entrypoint.py")
                else:
                    # SINGLE template is already entrypoint.py, remove multi_entrypoint.py
                    os.remove(f"{deployment_dir}/multi_entrypoint.py")
                    logger.info("Using single-agent template as entrypoint.py")
                
                # Step 3: Customize business logic using Bedrock
                logger.info("Step 3: Customizing business logic with Bedrock...")
                
                # Read the selected template content
                with open(f"{deployment_dir}/entrypoint.py", 'r') as f:
                    template_content = f.read()
                
                # Read the requirements.txt content
                with open(f"{deployment_dir}/requirements.txt", 'r') as f:
                    requirements_content = f.read()
                
                # Customize the business logic while preserving architecture
                try:
                    customized_entrypoint, updated_requirements = self.customize_business_logic(
                        agent_description, 
                        template_content, 
                        requirements_content
                    )
                    
                    # Write the customized files back
                    with open(f"{deployment_dir}/entrypoint.py", 'w') as f:
                        f.write(customized_entrypoint)
                    
                    with open(f"{deployment_dir}/requirements.txt", 'w') as f:
                        f.write(updated_requirements)
                    
                    logger.info("Successfully customized entrypoint.py and requirements.txt")
                    
                except Exception as customize_error:
                    logger.warning(f"Failed to customize business logic: {customize_error}")
                    logger.info("Proceeding with original template files")
                
                # Step 4: Deploy using agentcore CLI with streaming output
                # Get environment variables for agentcore CLI
                account = os.getenv('AWS_ACCOUNT', None)
                if not account:
                    # Try to get account from STS
                    try:
                        sts_client = boto3.client('sts')
                        account = sts_client.get_caller_identity()['Account']
                    except Exception:
                        raise Exception('AWS_ACCOUNT must be set in environment variables or AWS credentials must be available for STS.')
                
                region = os.getenv('REGION', os.getenv('AWS_REGION', os.getenv("AWS_DEFAULT_REGION", 'us-west-2')))
                
                # Change to the deployment directory
                original_cwd = os.getcwd()
                os.chdir(str(deployment_dir))
                
                try:
                    # Handle None values for CLI
                    if execution_role_arn is None:
                        execution_role_arn = 'None'
                    
                    if ecr_repo_uri is None:
                        ecr_repo_uri = 'None'

                    # Find the agentcore executable
                    import sys
                    
                    # Try to find the project's virtual environment
                    project_root = Path(__file__).parent.parent.parent
                    possible_venv_locations = [
                        project_root / '.venv' / 'bin',  # Standard .venv location
                        Path.cwd() / '.venv' / 'bin',    # Current directory .venv
                        Path(sys.executable).parent,     # Current Python bin directory
                    ]
                    
                    agentcore_exe = None
                    for venv_path in possible_venv_locations:
                        potential_exe = venv_path / 'agentcore'
                        if potential_exe.exists():
                            agentcore_exe = potential_exe
                            logger.info(f"Found agentcore executable at {agentcore_exe}")
                            break
                    
                    if not agentcore_exe:
                        # Final fallback - try to use which/where command
                        try:
                            which_result = shutil.which('agentcore')
                            if which_result:
                                agentcore_exe = Path(which_result)
                                logger.info(f"Found agentcore using which command: {agentcore_exe}")
                            else:
                                raise Exception(f"Could not find agentcore executable. Tried paths: {[str(p / 'agentcore') for p in possible_venv_locations]}. Also tried 'which agentcore' but not found.")
                        except Exception:
                            raise Exception(f"Could not find agentcore executable. Tried paths: {[str(p / 'agentcore') for p in possible_venv_locations]}")
                    
                    # Run agentcore configure
                    args = [
                        str(agentcore_exe), 'configure', 
                        '--name', name,
                        '--execution-role', execution_role_arn, 
                        '--ecr', ecr_repo_uri, 
                        '--entrypoint', entrypoint,
                        '--requirements-file', 'requirements.txt',
                        '--protocol', protocol,
                        '--region', region
                    ]

                    logger.info(f"Running agentcore configure command: {' '.join(args)}")
                    
                    # Use pipes for real-time streaming and provide stdin to answer OAuth prompt
                    process = subprocess.Popen(args, 
                                              stdout=subprocess.PIPE, 
                                              stderr=subprocess.STDOUT,
                                              stdin=subprocess.PIPE,
                                              text=True)
                    
                    # Provide "no" input to the OAuth prompt
                    try:
                        process.stdin.write("no\n")
                        process.stdin.flush()
                        process.stdin.close()
                    except Exception as e:
                        logger.info(f"Failed to write to stdin: {e}")
                    
                    # Stream output in real-time
                    stdout_data = ""
                    while True:
                        output = process.stdout.readline()
                        if output == '' and process.poll() is not None:
                            break
                        if output:
                            stdout_data += output
                            logger.info(f"agentcore configure: {output.strip()}")
                    
                    returncode = process.poll()
                    logger.info(f'Agentcore configure completed with return code {returncode}')
                    
                    if returncode != 0:
                        raise Exception(f"Error during agentcore configure.\noutput: {stdout_data}")
                    else:
                        logger.info('agentcore configure completed successfully.')

                    # Modify .bedrock_agentcore.yaml configuration
                    config_file = os.path.join(str(deployment_dir), '.bedrock_agentcore.yaml')
                    
                    with open(config_file, 'r') as config_in:
                        config_lines = config_in.readlines()
                        logger.info(f'Got config lines: {"".join(config_lines)}')
                        
                        final_config_lines = ''
                        for line in config_lines:
                            if 'execution_role: ' in line:
                                if not line.strip().endswith('None'):
                                    final_config_lines += line
                                else:
                                    logger.info(f"Skipping execution role None")
                            elif 'execution_role_auto_create: ' in line:
                                logger.info(f"execution role arn is {execution_role_arn}, type {type(execution_role_arn)}")
                                if execution_role_arn == 'None':
                                    line = line.replace('false', 'true')
                                final_config_lines += line
                            elif 'ecr_repository: ' in line:
                                if not line.strip().endswith('None'):
                                    final_config_lines += line
                            elif 'ecr_auto_create: ' in line:
                                logger.info(f"ecr_repo_uri == {ecr_repo_uri}, type {type(ecr_repo_uri)}")
                                if ecr_repo_uri == 'None':
                                    line = line.replace('false', 'true')
                                final_config_lines += line
                            else:
                                final_config_lines += line

                    logger.info(f"Final config lines before running agentcore launch: {final_config_lines}")
                    
                    with open(config_file, 'w') as config_out:
                        config_out.write(final_config_lines)
                        
                    # Run agentcore launch
                    launch_args = [str(agentcore_exe), 'launch']
                    if update_on_conflict:
                        launch_args.append('--auto-update-on-conflict')
                        
                    logger.info(f"Running agentcore launch command: {' '.join(launch_args)}")
                    
                    # Use pipes for real-time streaming
                    process = subprocess.Popen(launch_args, 
                                              stdout=subprocess.PIPE, 
                                              stderr=subprocess.STDOUT,
                                              text=True)
                    
                    # Stream output in real-time
                    stdout_data = ""
                    while True:
                        output = process.stdout.readline()
                        if output == '' and process.poll() is not None:
                            break
                        if output:
                            stdout_data += output
                            logger.info(f"agentcore launch: {output.strip()}")
                    
                    returncode = process.poll()
                    logger.info(f'Agentcore launch completed with return code {returncode}')
                    
                    if returncode != 0:
                        raise Exception(f"Error during agentcore launch.\noutput: {stdout_data}")

                    # Extract agent ARN from result
                    agent_arn = None
                    lines = stdout_data.split('\n')
                    for line in lines:
                        if 'Deployment completed successfully - Agent: ' in line:
                            logger.info(f"Found agent line: {line}")
                            agent_arn = line.split(' Agent: ')[1]
                            break
                            
                    if not agent_arn:
                        # Try looking for other patterns in output
                        for line in lines:
                            if 'Agent: arn:' in line:
                                logger.info(f"Found agent ARN in output: {line}")
                                agent_arn = line.split('Agent: ')[1].strip()
                                break
                    
                    if not agent_arn:
                        logger.warning("Could not extract agent ARN from agentcore launch output")
                        agent_arn = f"arn:aws:bedrock-agentcore:{region}:{account}:runtime/{name}"
                        
                    agent_runtime_id = agent_arn.split('/')[-1]
                    logger.info(f'Created agent runtime with ARN: {agent_arn}, ID: {agent_runtime_id}')
                    
                    # Get runtime details
                    try:
                        response = self.control_client.get_agent_runtime(agentRuntimeId=agent_runtime_id)
                        logger.info(f"Successfully retrieved runtime details for {agent_runtime_id}")
                        
                        return f"Successfully created agent runtime:\n\nRuntime Name: {name}\nRuntime ID: {agent_runtime_id}\nRuntime ARN: {agent_arn}\nStatus: {response.get('status', 'UNKNOWN')}\n\nDeployment Directory: {deployment_dir}\n\nAgent Description: {agent_description}\n\nYou can now invoke this runtime using the invoke_agent_runtime tool."
                        
                    except Exception as e:
                        logger.warning(f"Could not retrieve runtime details: {e}")
                        return f"Successfully created agent runtime:\n\nRuntime Name: {name}\nRuntime ID: {agent_runtime_id}\nRuntime ARN: {agent_arn}\n\nDeployment Directory: {deployment_dir}\n\nAgent Description: {agent_description}"
                    
                finally:
                    # Return to original directory
                    os.chdir(original_cwd)
                
            except Exception as e:
                # Clean up deployment directory on error
                try:
                    if Path(deployment_dir).exists():
                        shutil.rmtree(deployment_dir)
                        logger.info(f"Cleaned up deployment directory on error: {deployment_dir}")
                except Exception as cleanup_err:
                    logger.warning(f"Failed to cleanup deployment directory on error: {cleanup_err}")
                raise
            
        except Exception as e:
            logger.error(f"Error creating agent runtime: {str(e)}")
            raise Exception(f"Failed to create agent runtime: {str(e)}")
    

    @staticmethod
    def delete_agentcore_runtime(
        request: DeleteAgentRuntimeRequest
    ) -> DeleteAgentRuntimeResponse:
        """
        Delete an AgentCore Runtime resource.
        
        Args:
            request: DeleteAgentRuntimeRequest containing runtime ID
            
        Returns:
            DeleteAgentRuntimeResponse with deletion status
            
        Raises:
            Exception: If runtime deletion fails
        """
        logger.info(f"Deleting AgentCore Runtime with ID: {request.agent_runtime_id}")
        
        try:
            # Delete the agent runtime
            response = agentcore_control_client.delete_agent_runtime(
                agentRuntimeId=request.agent_runtime_id
            )
            
            logger.info(f"Successfully deleted AgentCore Runtime with ID: {request.agent_runtime_id}")
            
            return DeleteAgentRuntimeResponse(
                agent_runtime_id=request.agent_runtime_id,
                status="DELETING"
            )
            
        except Exception as e:
            logger.error(f"Error deleting AgentCore Runtime: {str(e)}")
            raise e

    @staticmethod
    def delete_tmpdir(tmpdir):
        return shutil.rmtree(tmpdir)
    
    @staticmethod
    def get_agentcore_runtime(
        request: GetAgentRuntimeRequest
    ) -> GetAgentRuntimeResponse:
        """
        Retrieve details of an AgentCore Runtime resource.
        
        Args:
            request: GetAgentRuntimeRequest containing runtime ID
            
        Returns:
            GetAgentRuntimeResponse with runtime details
            
        Raises:
            Exception: If runtime retrieval fails
        """
        agent_runtime_id = None
        if not hasattr(request, 'agent_runtime_id') and \
            hasattr(request, 'agent_runtime_arn'):
            agent_runtime_id = request.agent_runtime_arn.split('/')[-1]
        else:
            agent_runtime_id = request.agent_runtime_id

        logger.info(f"Getting AgentCore Runtime with ID: {agent_runtime_id}")
        
        try:
            # Get the agent runtime details
            response = agentcore_control_client.get_agent_runtime(
                agentRuntimeId=agent_runtime_id
            )
            if response['ResponseMetadata']['HTTPStatusCode'] != 200:
                return response
            
            logger.info(f"get_agent_runtime response {response}")

            response['createdAt'] = response['createdAt'].isoformat()
            response['lastUpdatedAt'] = response['lastUpdatedAt'].isoformat()
            del response['ResponseMetadata']
            logger.info(f"Response is now {response}")
            
            logger.info(f"Successfully retrieved AgentCore Runtime with ID: {agent_runtime_id}")
            
            return GetAgentRuntimeResponse(
                agent_runtime_arn=response['agentRuntimeArn'],
                agent_runtime_id=response['agentRuntimeId'],
                agent_runtime_version=response['agentRuntimeVersion'],
                agent_runtime_name=response['agentRuntimeName'],
                status=response['status'],
                workload_identity_details=response['workloadIdentityDetails'],
                created_at=response['createdAt'],
                last_updated_at=response['lastUpdatedAt'],
                role_arn=response['roleArn'],
                agent_runtime_artifact=response['agentRuntimeArtifact'],
                network_configuration=response['networkConfiguration'],
                protocol_configuration=response['protocolConfiguration'],
                authorizer_configuration=response['authorizerConfiguration']
            )
            
        except Exception as e:
            logger.error(f"Error getting AgentCore Runtime: {str(e)}")
            # Check if this is a ResourceNotFoundException that should return 404
            if hasattr(e, 'response') and 'Error' in e.response:
                error_code = e.response['Error'].get('Code', '')
                if error_code == 'ResourceNotFoundException':
                    from fastapi import HTTPException
                    raise HTTPException(status_code=404, detail=f"AgentCore Runtime not found: {agent_runtime_id}")
            raise e

    @staticmethod
    def get_tmpdir():
        tmpdir = f"/tmp/{uuid4().hex[:6]}"
        os.makedirs(tmpdir)
        return tmpdir
        

    @staticmethod
    def list_agent_runtimes(
        request: ListAgentRuntimesRequest
    ) -> ListAgentRuntimesResponse:
        """
        List AgentCore Runtime resources.
        
        Args:
            request: ListAgentRuntimesRequest containing listing parameters
            
        Returns:
            ListAgentRuntimesResponse with list of runtimes
            
        Raises:
            Exception: If runtime listing fails
        """
        logger.info(f"Listing AgentCore Runtimes got request {request}")
        
        try:
            # Prepare list parameters
            list_params = {}
            
            if request.max_results:
                list_params['maxResults'] = request.max_results
                
            if hasattr(request, 'next_token') and request.next_token:
                list_params['nextToken'] = request.next_token
            
            # List the agent runtimes
            response = agentcore_control_client.list_agent_runtimes(**list_params)
            logger.info(f"response: {response}")
            del response['ResponseMetadata']
            runtimes = []
            for runtime in response['agentRuntimes']:
                logger.info(f"Got runtime {runtime}")
                args = {
                    "agent_runtime_arn": runtime['agentRuntimeArn'],
                    "agent_runtime_id": runtime['agentRuntimeId'],
                    "agent_runtime_version": runtime['agentRuntimeVersion'],
                    "agent_runtime_name": runtime['agentRuntimeName'],
                    "status": runtime['status'] if isinstance(runtime['status'], str) else runtime['status'].value,
                }
                if hasattr(runtime,'lastUpdatedAt') and runtime.lastUpdatedAt:
                    args['last_updated_at'] = runtime.lastUpdatedAt.isoformat()
                
                runtimes.append(AgentRuntime(**args))
            logger.info(f"Successfully listed {len(runtimes)} AgentCore Runtimes")
            
            args = {
                "agent_runtimes": runtimes
            }
            if response.get('nextToken'):
                args['next_token'] = response.get('nextToken')

            return ListAgentRuntimesResponse(**args)
            
        except Exception as e:
            logger.error(f"Error listing AgentCore Runtimes: {str(e)}")
            raise e

    @staticmethod
    def update_agentcore_runtime(
        request: UpdateAgentRuntimeRequest
    ) -> UpdateAgentRuntimeResponse:
        """
        Update an AgentCore Runtime resource.
        
        Args:
            request: UpdateAgentRuntimeRequest containing update parameters
            
        Returns:
            UpdateAgentRuntimeResponse with updated runtime details
            
        Raises:
            Exception: If runtime update fails
        """
        logger.info(f"Updating AgentCore Runtime with request: {request}, type {type(request)}")
        # first get the old runtime details:
        old_runtime = None
        # logger.info(f"UpdateAgentRuntimeRequest attrs: {vars(request)}")
        # logger.info(f"Is there an agent_runtime_id property? {hasattr(request, 'agent_runtime_id')}")
        # logger.info(f"how about the vars way? vars(request)['agent_runtime_id'] = {vars(request)['agent_runtime_id']}")
        # logger.info(f"How about the property way? (next line crashes for some reason) request.agent_runtime_id = ")
        # logger.info(request.agent_runtime_id)
        agent_runtime_id = request.agent_runtime_id
        logger.info(f"Got agent_runtime_id {agent_runtime_id}")
        try: 

            old_runtime = AgentCoreRuntimeClient.get_agentcore_runtime(
                GetAgentRuntimeRequest(
                    agent_runtime_id=agent_runtime_id
                )
            )
            logger.info(f"Got old runtime {old_runtime}")
        except Exception as e:
            logger.info(f"Error updating runtime: {str(e)}")
            raise e
        try:
            # Prepare update parameters - only include non-None values
            # Only perform update if we have parameters to update
            update_request_params = {
                "agent_runtime_id": old_runtime.agent_runtime_id,
                "agentRuntimeArtifact": request.agentRuntimeArtifact,
                "roleArn": request.roleArn,
                "networkConfiguration": request.networkConfiguration,
                "protocolConfiguration": request.protocolConfiguration,
                "authorizerConfiguration": request.authorizerConfiguration
            }
            if hasattr(request, 'clientToken') and request.clientToken:
                update_request_params['clientToken'] = request.clientToken
            if hasattr(request, 'description') and request.description:
                update_request_params['description'] = request.description
            else:
                update_request_params['description'] = old_runtime['description']
            if hasattr(request, 'environmentVariables') and request.environmentVariables:
                update_request_params['environmentVariables'] = request.environmentVariables
            else:
                update_request_params['environmentVariables'] = old_runtime.environmentVariables

            logger.info(f"Update params currently {update_request_params}")
            # Update the agent runtime
            response = agentcore_control_client.update_agent_runtime(**update_request_params)
            response['createdAt'] = response['createdAt'].isoformat()
            response['lastUpdatedAt'] = response['lastUpdatedAt'].isoformat()
            logger.info(f"Successfully updated AgentCore Runtime with ID: {request.agent_runtime_id}")
            logger.info(response)
            return response
           
        except Exception as e:
            logger.error(f"Error updating AgentCore Runtime: {str(e)}")
            raise e

    @staticmethod
    def wait_for_runtime_ready(
        agent_runtime_id: str,
        max_wait_time: int = 600,
        poll_interval: int = 10
    ) -> GetAgentRuntimeResponse:
        """
        Wait for an AgentCore Runtime to finish creating and become ready or fail.
        
        Polls the runtime status until it's no longer in 'CREATING' state.
        Returns when status becomes 'READY', or 'FAILED'.
        
        Args:
            agent_runtime_id: The ID of the agent runtime to monitor
            max_wait_time: Maximum time to wait in seconds (default: 600 = 10 minutes)
            poll_interval: Time to wait between status checks in seconds (default: 10)
            
        Returns:
            GetAgentRuntimeResponse with the final runtime details
            
        Raises:
            TimeoutError: If runtime doesn't reach a final state within max_wait_time
            Exception: If runtime retrieval fails
        """
        logger.info(f"Waiting for AgentCore Runtime {agent_runtime_id} to be ready...")
        
        start_time = time.time()
        creating_states = [
            AgentRuntimeStatus.CREATING, 
            AgentRuntimeStatus.UPDATING
        ]
        ready_states = [
            AgentRuntimeStatus.READY,
        ]
        failed_states = [
            AgentRuntimeStatus.CREATE_FAILED, 
            AgentRuntimeStatus.UPDATE_FAILED
        ]
        
        while True:
            try:
                # Get current runtime status
                get_request = GetAgentRuntimeRequest(agent_runtime_id=agent_runtime_id)
                runtime_response = AgentCoreRuntimeClient.get_agentcore_runtime(get_request)
                
                current_status = runtime_response.status
                logger.info(f"Current status for runtime {agent_runtime_id}: {current_status}")
                
                # Check if runtime is ready
                if current_status in ready_states:
                    logger.info(f"AgentCore Runtime {agent_runtime_id} is ready with status: {current_status}")
                    return runtime_response
                
                # Check if runtime failed
                if current_status in failed_states:
                    error_msg = f"AgentCore Runtime {agent_runtime_id} failed with status: {current_status}"
                    logger.error(error_msg)
                    raise Exception(error_msg)
                
                # Check if we've exceeded the maximum wait time
                elapsed_time = time.time() - start_time
                if elapsed_time >= max_wait_time:
                    error_msg = f"Timeout waiting for AgentCore Runtime {agent_runtime_id} to be ready. Current status: {current_status}, elapsed time: {elapsed_time:.1f}s"
                    logger.error(error_msg)
                    raise TimeoutError(error_msg)
                
                # Runtime is still creating/updating, wait before next check
                if current_status in creating_states:
                    logger.info(f"Runtime {agent_runtime_id} still {current_status.lower()}, waiting {poll_interval}s before next check...")
                    time.sleep(poll_interval)
                else:
                    # Unexpected status, log warning but continue waiting
                    logger.warning(f"Runtime {agent_runtime_id} has unexpected status: {current_status}, continuing to wait...")
                    time.sleep(poll_interval)
                    
            except Exception as e:
                # If it's a timeout error we raised, re-raise it
                if isinstance(e, TimeoutError):
                    raise e
                
                # For other exceptions, log and re-raise
                logger.error(f"Error checking runtime status: {str(e)}")
                raise Exception(f"Failed to check runtime status: {str(e)}")

    @staticmethod
    def sanitize_name(original_name: str) -> str:
        """Sanitize agent name to meet AgentCore requirements.
        
        AgentCore agent names must:
        - Start with a letter
        - Contain only letters, numbers, and underscores  
        - Be 1-48 characters long
        
        Args:
            original_name: The original agent name to sanitize
            
        Returns:
            Sanitized agent name that meets AgentCore requirements
        """
        import re
        
        # Replace invalid characters with underscores
        sanitized_name = re.sub(r'[^a-zA-Z0-9_]', '_', original_name)
        
        # Ensure it starts with a letter
        if not sanitized_name or not sanitized_name[0].isalpha():
            sanitized_name = 'agent_' + sanitized_name
        
        # Truncate to 48 characters if needed
        if len(sanitized_name) > 48:
            sanitized_name = sanitized_name[:48]
        
        # Remove trailing underscores that might result from truncation
        sanitized_name = sanitized_name.rstrip('_')
        
        # Ensure it's not empty after sanitization
        if not sanitized_name:
            sanitized_name = 'agent_runtime'
            
        if original_name != sanitized_name:
            logger.info(f"Sanitized agent name from '{original_name}' to '{sanitized_name}'")
        
        return sanitized_name

    @staticmethod
    def select_entrypoint_template(user_description: str) -> str:
        """Given the user's request, decide if the single or multi-agent entrypoint is needed.
        
        Uses Bedrock to analyze the user's description and determine whether a single
        agent or multi-agent orchestration template would be more appropriate.
        
        Args:
            user_description: Natural language description of what the agent should do
            
        Returns:
            'SINGLE' or 'MULTI' indicating which template to use
        """
        try:
            # Construct the prompt for the LLM
            prompt_template = """You're an agentic AI architect. Given the user's request, does it sound like they'll need a single agent or multi-agent template for this project?
            
            <USER_DESCRIPTION>
            {user_description}
            </USER_DESCRIPTION> 

            Return only the word SINGLE or MULTI with no other text or newlines.
            """

            formatted_prompt = prompt_template.format(user_description=user_description)
            
            # Prepare the message for Bedrock Converse API
            messages = [
                {
                    "role": "user",
                    "content": [{"text": formatted_prompt}]
                }
            ]
            logger.info("Sending entrypoint selection prompt to bedrock")
            
            # Call Bedrock Converse API with Nova Micro
            response = bedrock_runtime_client.converse(
                modelId="us.amazon.nova-micro-v1:0",
                messages=messages,
                inferenceConfig={
                    "maxTokens": 50,
                    "temperature": 0.0,  # Lower temperature for more consistent outputs
                    "topP": 0.9,
                    "stopSequences": ["</JSON>"]
                }
            )
            logger.info(f"got response from bedrock {response}")

            # Extract the generated selection
            result = response['output']['message']['content'][0]['text'].strip()
            logger.info(f"entrypoint selection result: {result}")
            
            # Validate result
            if result in ['SINGLE', 'MULTI']:
                return result
            else:
                logger.warning(f"Unexpected template selection result: {result}, defaulting to SINGLE")
                return 'SINGLE'
            
        except Exception as e:
            logger.error(f"Failed to select template via Bedrock: {e}")
            logger.info("Defaulting to SINGLE template")
            return 'SINGLE'


### Set up the environment and import required modules

In [ ]:
# first make sure the sample-agentic-platform/src is in our path

import sys
sys.path

In [ ]:
# if it's not, add it

sys.path.insert(0, '../../src')
sys.path

### Now let's create an AgentCore Runtime

First we need to set up our runtime client and get configuration from our Terraform deployment

In [ ]:
from agentic_platform.service.agentcore.runtime.client.agentcore_runtime_client import AgentCoreRuntimeClient
from agentic_platform.service.agentcore.types import (
    GetAgentRuntimeRequest,
    ListAgentRuntimesRequest,
    DeleteAgentRuntimeRequest
)

import json
import os

# Set environment variables for the runtime client
os.environ['REGION'] = tf_info['aws_region']['value']
os.environ['USER_POOL_CLIENT_ID'] = tf_info['cognito_user_pool_client_id']['value']
os.environ['USER_POOL_ID'] = tf_info['cognito_user_pool_id']['value']

# Create the runtime client
runtime_client = AgentCoreRuntimeClient()

print(f"Runtime client configured for region: {tf_info['aws_region']['value']}")
print(f"Using Cognito User Pool: {tf_info['cognito_user_pool_id']['value']}")

### Create a test agent runtime with custom business logic

The AgentCore Runtime Client will automatically:
1. Generate custom business logic based on the description using Bedrock
2. Create deployment artifacts
3. Deploy the runtime using the AgentCore CLI
4. Return the runtime details

In [ ]:
%store -r runtime_id
agent_runtime_id = None

try:
    agent_runtime_id = runtime_id
    # Create a test agent runtime
except NameError:
    print("Creating new AgentRuntimeClient.")
    runtime_client = AgentCoreRuntimeClient()
    runtime_response = runtime_client.create_agent_runtime(
        name='agentpath-labs-module6-test-runtime',
        agent_description='A helpful assistant agent that can answer questions about AWS services, provide code examples, and help with troubleshooting. The agent should be knowledgeable about cloud architecture and best practices.',
        protocol='HTTP',
        update_on_conflict=True
    )

    print("Runtime Creation Response:")
    print(runtime_response)

    # Extract runtime ID from the response
    runtime_lines = runtime_response.split('\n')
    runtime_id = None
    runtime_arn = None

    for line in runtime_lines:
        if 'Runtime ID:' in line:
            runtime_id = line.split('Runtime ID: ')[1].strip()
        elif 'Runtime ARN:' in line:
            runtime_arn = line.split('Runtime ARN: ')[1].strip()

    print(f"\nCreated runtime with ID: {runtime_id}")
    print(f"Runtime ARN: {runtime_arn}")
    agent_runtime_id = runtime_id
    print(f"Storing {agent_runtime_id} for future use")
    %store runtime_id


### Now let's retrieve the runtime details

In [ ]:
# Get runtime details
if agent_runtime_id:
    get_request = GetAgentRuntimeRequest(agent_runtime_id=agent_runtime_id)
    runtime_details = runtime_client.get_agentcore_runtime(get_request)
    
    print("Runtime Details:")
    print(f"Name: {runtime_details.agent_runtime_name}")
    print(f"ID: {runtime_details.agent_runtime_id}")
    print(f"ARN: {runtime_details.agent_runtime_arn}")
    print(f"Status: {runtime_details.status}")
    print(f"Version: {runtime_details.agent_runtime_version}")
    print(f"Created: {runtime_details.created_at}")
    print(f"Last Updated: {runtime_details.last_updated_at}")
    print(f"Role ARN: {runtime_details.role_arn}")
    print(f"Workload Identity: {runtime_details.workload_identity_details}")
else:
    print("No runtime ID available - runtime creation may have failed")

### Let's list all runtimes to see our new runtime

In [ ]:
# List all runtimes
list_request = ListAgentRuntimesRequest(max_results=10)
runtimes_list = runtime_client.list_agent_runtimes(list_request)

print(f"Found {len(runtimes_list.agent_runtimes)} runtimes:")
for i, runtime in enumerate(runtimes_list.agent_runtimes, 1):
    print(f"{i}. Name: {runtime.agent_runtime_name}")
    print(f"   ID: {runtime.agent_runtime_id}")
    print(f"   Status: {runtime.status}")
    print(f"   Version: {runtime.agent_runtime_version}")
    if hasattr(runtime, 'last_updated_at') and runtime.last_updated_at:
        print(f"   Last Updated: {runtime.last_updated_at}")
    

### Wait for the runtime to be ready (if it's still creating)

Runtime creation can take several minutes as it involves:
1. Building the container image
2. Pushing to ECR
3. Deploying to the AgentCore platform
4. Health checks and readiness verification

In [ ]:
# Wait for runtime to be ready if it's still creating
if runtime_id and runtime_details.status in ['CREATING', 'UPDATING']:
    print(f"Runtime is {runtime_details.status.lower()}, waiting for it to be ready...")
    print("This may take several minutes...")
    
    try:
        ready_runtime = runtime_client.wait_for_runtime_ready(
            agent_runtime_id=runtime_id,
            max_wait_time=600,  # 10 minutes
            poll_interval=30    # Check every 30 seconds
        )
        
        print(f"\n✅ Runtime is now ready!")
        print(f"Final Status: {ready_runtime.status}")
        print(f"Runtime ARN: {ready_runtime.agent_runtime_arn}")
        
    except TimeoutError as e:
        print(f"⏰ Timeout waiting for runtime: {e}")
        print("Runtime may still be creating - check AWS console for status")
    except Exception as e:
        print(f"❌ Error waiting for runtime: {e}")
        
elif runtime_id:
    print(f"Runtime is already in status: {runtime_details.status}")
else:
    print("No runtime ID available to wait for")

### Now let's demonstrate error handling by trying to get a non-existent runtime

In [ ]:
# Test error handling with non-existent runtime
try:
    non_existent_request = GetAgentRuntimeRequest(agent_runtime_id='non-existent-runtime-id')
    non_existent_runtime = runtime_client.get_agentcore_runtime(non_existent_request)
    print("This should not print")
except Exception as e:
    print(f"Expected error when trying to get non-existent runtime: {str(e)[:100]}...")

### Let's create another runtime with different configuration to demonstrate multiple runtimes

In [ ]:
%store -r second_runtime_id

agent_runtime_id2 = None
try: 
    agent_runtime_id2 = second_runtime_id
    print(f"Got second runtime id from storage {agent_runtime_id2}")
except NameError:
    print(f"Creating a new runtime.")
    # Create a second runtime with different configuration
    runtime_response_2 = runtime_client.create_agent_runtime(
        name='agentpath-labs-module6-data-runtime',
        agent_description='A specialized data analysis agent that can process CSV files, generate visualizations, perform statistical analysis, and create reports. The agent should be skilled in pandas, matplotlib, and data science workflows.',
        protocol='HTTP',
        update_on_conflict=True
    )

    print("Second Runtime Creation Response:")
    print(runtime_response_2)

    # Extract runtime ID from the response
    runtime_lines_2 = runtime_response_2.split('\n')
    second_runtime_id = None

    for line in runtime_lines_2:
        if 'Runtime ID:' in line:
            second_runtime_id = line.split('Runtime ID: ')[1].strip()
            break

    print(f"\nSecond runtime ID: {second_runtime_id}")
    %store second_runtime_id 
    agent_runtime_id2 = second_runtime_id



### Now let's list all runtimes again to see both

In [ ]:
# List all runtimes again
all_runtimes = runtime_client.list_agent_runtimes(ListAgentRuntimesRequest())

print(f"\nTotal runtimes found: {len(all_runtimes.agent_runtimes)}")
print("\nRuntime Summary:")
for i, runtime in enumerate(all_runtimes.agent_runtimes, 1):
    print(f"{i}. Name: {runtime.agent_runtime_name}")
    print(f"   ID: {runtime.agent_runtime_id}")
    print(f"   Status: {runtime.status}")
    print(f"   ARN: {runtime.agent_runtime_arn}")
    print(f"   Version: {runtime.agent_runtime_version}")
    if hasattr(runtime, 'last_updated_at') and runtime.last_updated_at:
        print(f"   Last Updated: {runtime.last_updated_at}")
    print()

### Let's demonstrate runtime update operations

Note: Runtime updates typically involve updating configuration, environment variables, or other runtime settings. The actual business logic updates would require redeployment.

In [ ]:
# Demonstrate runtime update. This code performs runtime updates using the AgentCore client
import time
import json
from datetime import datetime
from agentic_platform.service.agentcore.types import UpdateAgentRuntimeRequest, GetAgentRuntimeRequest

runtime_id = agent_runtime_id


if runtime_id:
    print(f"Performing runtime update operations on runtime: {runtime_id}")
    
    # Get current runtime details for update
    current_runtime = runtime_client.get_agentcore_runtime(
        GetAgentRuntimeRequest(agent_runtime_id=runtime_id)
    )
    print(f"Got current runtime: {json.dumps(current_runtime.to_dict(), indent=2)}")
    print(f"\nCurrent runtime status: {current_runtime.status}")
    print(f"Current runtime version: {current_runtime.agent_runtime_version}")
    print(f"Current runtime name: {current_runtime.agent_runtime_name}")
    
    # Example 1: Update runtime with new environment variables
    print("\n=== Updating Runtime Environment Variables ===")
    
    # Prepare update request with new environment variables
    update_request = UpdateAgentRuntimeRequest(
        agent_runtime_id=runtime_id,
        environmentVariables={
            "UPDATED_TIMESTAMP": str(int(time.time())),
            "UPDATE_VERSION": "v1.1",
            "ENVIRONMENT": "development"
        }
    )
    
    # Perform the update - LET EXCEPTIONS RAISE
    update_response = runtime_client.update_agentcore_runtime(update_request)
    runtime_client.wait_for_runtime_ready(runtime_id)
    print(f"✅ Runtime update initiated successfully!")
    print(f"Updated runtime status: {update_response.get('status')}")
    print(f"Last updated: {update_response.get('lastUpdatedAt', 'N/A')}")
    
    # Example 2: Update runtime description
    print("\n=== Updating Runtime Description ===")
    
    update_request = UpdateAgentRuntimeRequest(
        agent_runtime_id=runtime_id,
        description=f"Updated runtime description - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
    )
    
    # Perform the update - LET EXCEPTIONS RAISE
    update_response = runtime_client.update_agentcore_runtime(update_request)
    print(f"Response from update: {update_response}")
    runtime_client.wait_for_runtime_ready(runtime_id)
    print(f"✅ Runtime description updated successfully!")
    print(f"Updated runtime status: {update_response.get('status')}")
    
    # Example 3: Wait for runtime to be ready after updates
    print("\n=== Waiting for Runtime to be Ready ===")
    
    # Wait for the runtime to finish updating - LET EXCEPTIONS RAISE
    final_runtime = runtime_client.wait_for_runtime_ready(
        agent_runtime_id=runtime_id,
        max_wait_time=300,  # 5 minutes
        poll_interval=10    # Check every 10 seconds
    )
    
    print(f"✅ Runtime is now ready!")
    print(f"Final status: {final_runtime.status}")
    print(f"Final version: {final_runtime.agent_runtime_version}")
    
    # Example 4: Advanced update with network configuration
    print("\n=== Advanced Runtime Configuration Update ===")
    
    # Example of updating network configuration (if supported)
    advanced_update_request = UpdateAgentRuntimeRequest(
        agent_runtime_id=runtime_id,
        networkConfiguration={
            "subnetIds": [],  # Add your subnet IDs if needed
            "securityGroupIds": []  # Add your security group IDs if needed
        },
        environmentVariables={
            "ADVANCED_CONFIG": "enabled",
            "LAST_UPDATED": datetime.now().isoformat()
        }
    )
    
    # Perform advanced update - LET EXCEPTIONS RAISE
    update_response = runtime_client.update_agentcore_runtime(advanced_update_request)
    runtime_client.wait_for_runtime_ready(runtime_id)

    print(f"✅ Advanced runtime configuration updated! {update_response}")
    print(f"Status: {update_response.get('status')}")
    
    print("\n=== Runtime Update Operations Complete ===")
    print("All supported runtime update operations have been demonstrated.")
    print("Note: Some updates may require the runtime to restart, which can take several minutes.")
    
else:
    raise ValueError("No runtime ID available for update operations. Please ensure you have created a runtime first using the create_agent_runtime method")


### Now let's clean up by deleting the test runtimes

**Important**: Deleting runtimes will permanently remove them and any associated resources. Make sure you don't need them before proceeding.

In [ ]:
# Delete the first runtime
if agent_runtime_id:
    print(f"Deleting runtime: {agent_runtime_id}")
    delete_request_1 = DeleteAgentRuntimeRequest(agent_runtime_id=agent_runtime_id)
    delete_response_1 = runtime_client.delete_agentcore_runtime(delete_request_1)
    print(f"Delete response: {delete_response_1.agent_runtime_id} - Status: {delete_response_1.status}")
else:
    print("No first runtime to delete")

# Delete the second runtime
if agent_runtime_id2:
    print(f"\nDeleting runtime: {agent_runtime_id2}")
    delete_request_2 = DeleteAgentRuntimeRequest(agent_runtime_id=agent_runtime_id2)
    delete_response_2 = runtime_client.delete_agentcore_runtime(delete_request_2)
    print(f"Delete response: {delete_response_2.agent_runtime_id} - Status: {delete_response_2.status}")
else:
    print("No second runtime to delete")

### Let's verify the runtimes were deleted

In [ ]:
# Wait a moment for deletion to process
import time
print("Waiting for deletion to complete...")
time.sleep(10)
runtime_id_2 = second_runtime_id
# Try to get the deleted runtimes (should fail)
for rt_id in [runtime_id, runtime_id_2]:
    if rt_id:
        try:
            deleted_request = GetAgentRuntimeRequest(agent_runtime_id=rt_id)
            deleted_runtime = runtime_client.get_agentcore_runtime(deleted_request)
            print(f"Runtime {rt_id} still exists with status: {deleted_runtime.status}")
        except Exception as e:
            print(f"Runtime {rt_id} successfully deleted (expected error: {str(e)[:50]}...)")

# List runtimes to confirm cleanup
final_runtimes = runtime_client.list_agent_runtimes(ListAgentRuntimesRequest())
remaining_test_runtimes = [
    rt for rt in final_runtimes.agent_runtimes 
    if 'agentpath-labs-module6' in rt.agent_runtime_name
]

print(f"\nRemaining test runtimes: {len(remaining_test_runtimes)}")
if remaining_test_runtimes:
    for rt in remaining_test_runtimes:
        print(f"- {rt.agent_runtime_name} ({rt.agent_runtime_id}) - Status: {rt.status}")
else:
    print("All test runtimes successfully cleaned up!")

## Summary

### What You've Accomplished

✅ **Agent Runtime Resource Management Operations**:
- **CREATE**: Created new agent runtimes with custom business logic generated by Bedrock
- **READ**: Retrieved runtime details, status, and configuration
- **UPDATE**: Explored runtime update capabilities and patterns
- **DELETE**: Cleaned up runtime resources
- **LIST**: Enumerated all runtime resources

✅ **Runtime Management Concepts**:
- Understood agent runtime lifecycle and deployment process
- Learned about automatic business logic generation using Bedrock
- Explored runtime status monitoring and readiness checks
- Practiced runtime configuration and management

✅ **Production Patterns**:
- Error handling and validation
- Resource cleanup procedures
- Proper runtime lifecycle management
- Wait operations for async deployment processes

### Key Learnings

1. **Agent Runtimes**: Provide managed execution environments for AI agents with automatic scaling and deployment
2. **Business Logic Generation**: Bedrock can automatically generate custom agent code based on natural language descriptions
3. **Deployment Process**: Runtime creation involves containerization, ECR deployment, and health checks
4. **Status Management**: Runtimes have complex lifecycle states that require monitoring and wait operations
5. **Resource Management**: Proper cleanup prevents resource accumulation and cost optimization

### Next Steps

- **Runtime Invocation**: Learn how to invoke deployed runtimes with requests and receive responses
- **Advanced Configuration**: Explore custom containers, environment variables, and scaling settings
- **Integration Patterns**: Connect runtimes with MCP gateways and other AgentCore services
- **Production Deployment**: Implement CI/CD pipelines for runtime deployment and updates

---

In [ ]:
%store -d runtime_id
%store -d second_runtime_id